In [ ]:
# Delta tables - SCD (slowly changing dimenstions):

# Slowly Changing Dimensions (SCD) are techniques in data warehousing to handle changes in dimension attributes over time.
# The most common types are SCD Type 0 through Type 4, each balancing history tracking, simplicity, and storage.

# Overview of SCD Types:

# Type 0 - 
#         Name: Fixed Dimension
#         Behavior: No changes allowed; values remain static.
#         Use Case: Immutable attributes like Date of Birth, SSN, Zip Code.

# Type 1 - 
#         Name: Overwrite
#         Behavior: Old value is overwritten with new value; no history kept.
#         Use Case: Correcting errors or when history is not important.
#         Practical Example: If a customer’s phone number changes, you simply update the record. Old number is lost.

# Type 2 - 
#         Name: Add New Row
#         Behavior: A new row is inserted with a surrogate key; history preserved with effective dates or flags.
#         Use Case: Tracking customer address changes, employee role changes.
#         Practical Example: If a customer moves to a new city, you insert a new row with start/end dates, preserving the old city for historical reporting.

# Type 3 - 
#         Name: Add New Column
#         Behavior: A new column stores the previous value; limited history (usually one prior state).
#         Use Case: When only current and one previous value are needed (e.g., last vs current department).
#         Practical Example: If you only need to know the current and previous city, you add a previous_city column.

# Type 4 - 
#         Name: History Table
#         Behavior: A separate history table stores all changes; main table keeps current value.
#         Use Case: When detailed history is needed but main table must stay lean.
#         Practical Example: Keep current city in the main table, but log all changes in a separate history table.

# Type 6 - 
#         Name: Hybrid (1+2+3)
#         Behavior: Combines overwrite, new row, and new column approaches.
#         Use Case: Complex scenarios requiring both current and historical tracking.

# Key Considerations:

#            Performance vs. History: Type 1 is fastest but loses history; Type 2 is most common for full historical tracking.
#            Storage: Type 2 and Type 4 increase storage requirements.
#            Query Complexity: Type 2 queries can be more complex due to multiple rows per entity.
#            Business Needs: Choose based on whether you need to answer “what is” (Type 1) or “what was” (Type 2/4).

# In practice, most enterprise data warehouses (including Fabric, Synapse, and Databricks) rely heavily on SCD Type 2 for historical accuracy, sometimes combined with Type 1 for corrections.

In [7]:
df=spark.read.csv('abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/cust_01052025.csv',header=True)
display(df)

df1=spark.read.csv('abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/cust_01062025.csv',header=True)
display(df1)

df2=spark.read.csv('abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/cust_july.csv',header=True)
display(df2)

StatementMeta(, 9fb415df-c862-4359-8353-ff6d430b16ea, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54713e12-f9ab-41bf-9c61-bce820f964f1)

SynapseWidget(Synapse.DataFrame, cca26c78-d9dc-4d7b-982c-823eaf591e4a)

SynapseWidget(Synapse.DataFrame, 3988009f-ed38-4231-88fe-f5be81ca6e0c)

In [4]:
%%sql

use spark_catalog;
create DATABASE if not exists db_source;
create DATABASE if not exists db_stage;
create DATABASE if not exists db_target;

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 9, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [11]:
#Create source table
df.write.format('delta').mode('overwrite').saveAsTable('db_source.cust_dtls');

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 17, Finished, Available, Finished, False)

In [39]:
%%sql
-- view source table data
SELECT * from db_source.cust_dtls;

-- insert source table as it is into stage table
insert into db_stage.cust_dtls
SELECT * from db_source.cust_dtls;

-- View stage table data
select * from db_stage.cust_dtls;


StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 64, Finished, Available, Finished, True)

<Spark SQL result set with 3 rows and 3 fields>

<Spark SQL result set with 3 rows and 3 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 3 rows and 3 fields>

In [41]:
%%sql
-- Create target data table with data
create table if not exists db_target.cust_dtls
(
    cust_id string,
    cust_name string,
    address string
)using delta;


-- --insert initial data into taget from stage table
-- insert into db_target.cust_dtls
-- select * from db_stage.cust_dtls;

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 68, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [42]:
%%sql
select * from db_target.cust_dtls;

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 69, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 3 fields>

In [44]:
%%sql
-- Insert data into target table
merge into db_target.cust_dtls as tgt
using db_stage.cust_dtls as stg
on tgt.cust_id=stg.cust_id
when matched and tgt.address<>stg.address then update set *
when not matched then insert * 

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 72, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [45]:
%%sql
select * from db_target.cust_dtls;

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 73, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 3 fields>

In [48]:
#nsert new data into source table
df1.write.format('delta').mode('overwrite').saveAsTable('db_source.cust_dtls');

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 76, Finished, Available, Finished, False)

In [52]:
%%sql
SELECT * FROM db_source.cust_dtls;

delete from db_stage.cust_dtls;
insert into db_stage.cust_dtls
select * from db_source.cust_dtls;

select * from db_stage.cust_dtls;

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 86, Finished, Available, Finished, True)

<Spark SQL result set with 4 rows and 3 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 4 rows and 3 fields>

In [53]:
%%sql
-- Insert data into target table
merge into db_target.cust_dtls as tgt
using db_stage.cust_dtls as stg
on tgt.cust_id=stg.cust_id
when matched and tgt.address<>stg.address then update set *
when not matched then insert * ;

-- View new data
SELECT * FROM db_target.cust_dtls;

StatementMeta(, d9240307-7505-41ae-aecf-5620d4c091f5, 88, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 4 rows and 3 fields>

In [13]:
#insert new data into source table - SCD Type2 Example
df2.write.format('delta').mode('overwrite').saveAsTable('db_source.cust_dtls');

StatementMeta(, 9fb415df-c862-4359-8353-ff6d430b16ea, 22, Finished, Available, Finished, False)

In [15]:
%%sql

delete from db_stage.cust_dtls;

insert into db_stage.cust_dtls
select * from db_source.cust_dtls;

select * from db_stage.cust_dtls;

StatementMeta(, 9fb415df-c862-4359-8353-ff6d430b16ea, 26, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 5 rows and 3 fields>

In [16]:
%%sql

-- Create target data table - Type2
create table if not exists db_target.cust_dtls_type2
(
    cust_id string,
    cust_name string,
    address string,
    active string,
    start_date date,
    end_date string

)using delta;

-- delete from db_target.cust_dtls_type2;
insert into db_target.cust_dtls_type2
select *,'Y',CURRENT_DATE(),NULL from db_target.cust_dtls;

SELECT * FROM db_target.cust_dtls_type2;
SELECT * FROM db_stage.cust_dtls;

StatementMeta(, 9fb415df-c862-4359-8353-ff6d430b16ea, 29, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 4 rows and 6 fields>

<Spark SQL result set with 5 rows and 3 fields>

In [23]:
%%sql
  select cs.cust_id as mergekey, cs.* from db_stage.cust_dtls as cs
    union all
    select NULL as mergekey,cs.* from db_stage.cust_dtls as cs
    join db_target.cust_dtls_type2 as t
    on t.cust_id=cs.cust_id and t.address <> cs.address and t.active='Y'

StatementMeta(, 9fb415df-c862-4359-8353-ff6d430b16ea, 36, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 4 fields>

In [25]:
%%sql
merge into db_target.cust_dtls_type2 as tgt
using (
    select cs.cust_id as mergekey, cs.* from db_stage.cust_dtls as cs
    union all
    select NULL as mergekey,cs.* from db_stage.cust_dtls as cs
    join db_target.cust_dtls_type2 as t
    on t.cust_id=cs.cust_id and t.address <> cs.address and t.active='Y'
) as stg
on tgt.cust_id=stg.mergekey and tgt.active='Y'
when matched and tgt.address<>stg.address then update set tgt.active='N', tgt.end_date=current_date()-1
when not matched then insert (tgt.cust_id, tgt.cust_name,tgt.address,tgt.active, tgt.start_date,tgt.end_date)
values(stg.cust_id,stg.cust_name,stg.address,'Y',current_date(),NULL);

select * from db_target.cust_dtls_type2;

StatementMeta(, 9fb415df-c862-4359-8353-ff6d430b16ea, 39, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 6 rows and 6 fields>